# Monte Carlo Simulation: OLS vs Newey–West HAC under GARCH(1,1) Autocorrelation

Goal: Evaluate the performance of ordinary least squares (OLS) standard errors
compared to heteroskedasticity-and-autocorrelation consistent (HAC)
Newey–West estimators when residuals follow a GARCH(1,1) process.


In [304]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac
from scipy.stats import norm, chi2, t
from tqdm.notebook import trange, tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats

In [305]:
# Simulation parameters
np.random.seed(2025)
R = 1000                # number of replications
T = 200               # sample size
betas_true = np.array([0.0, 1.0, 0.5, -0.5])
bandwidth = int(4 * (T/100)**(2/9))  # HAC lag lengths
# GARCH(1,1) parameters
omega_garch = 0.01
alpha_garch = 0.13
beta_garch = 0.83
k = len(betas_true)  # number of parameters


In [306]:
def simulate_garch(T, omega, alpha, beta):
    """ Simulate GARCH(1,1) process """
    eps = np.zeros(T)
    h = np.zeros(T)
    h[0] = omega / (1 - alpha - beta)  # unconditional variance
    eps[0] = np.sqrt(h[0]) * np.random.normal()
    for t in range(1, T):
        h[t] = omega + alpha * eps[t-1]**2 + beta * h[t-1]
        eps[t] = np.sqrt(h[t]) * np.random.normal()
    return eps
def one_replication(T, betas_true, bandwidth, omega_garch, alpha_garch, beta_garch):
    """ Run one Monte Carlo replication comparing OLS and Newey-West standard errors.
    Returns a dictionary with:
    betahat, residuals, residuals_standarized, s2, R2, R2_adj, cov_betas, se_beta, t_stats, p_value, conf_int_95
    for both OLS and HAC.
    """
    # 1. Simulate regressors
    X = np.column_stack([
        np.ones(T),
        np.random.normal(size=T),
        np.random.normal(size=T),
        np.random.normal(size=T)
    ])
    # --- 2️⃣ Simulate GARCH(1,1) errors ---
    u = simulate_garch(T,omega_garch, alpha_garch, beta_garch)

    # --- 3️⃣ Generate dependent variable ---
    y = np.matmul(X,betas_true) + u

    # --- 4️⃣ Fit OLS model ---
    model = sm.OLS(y, X).fit()
    betahat = np.linalg.inv(X.T @ X) @ X.T @ y
    k = len(betahat)
    df = T - k

    # --- 5️⃣ Initialize results dictionary ---
    results = {}
    # --- 6️⃣ Compute OLS quantities ---

    residuals = (y - X @ betahat)
    SSR = np.dot(residuals,residuals)
    SST = np.dot(y - np.mean(y), y -np.mean(y))
    R2_ols = 1 - SSR / SST
    R2_adj_ols = 1 - (SSR / df) / (SST / (T-1))
    s2_ols = SSR / df
    cov_beta_ols = s2_ols * np.linalg.inv(X.T @ X)
    se_beta_ols = np.sqrt(np.diag(cov_beta_ols))
    t_stats_ols = betahat / se_beta_ols
    p_value_ols = 2 * (1 - t.cdf(np.abs(t_stats_ols),df))
    z_95 = norm.ppf(1-0.05/2)
    ci_asymp_ols_95 = np.column_stack([betahat - z_95*se_beta_ols, betahat + z_95*se_beta_ols])
    residuals_studentized = residuals / (np.sqrt(s2_ols) * np.sqrt(1 - np.diag(X @ np.linalg.inv(X.T @ X) @ X.T))) 

    results['OLS'] = {
        'betahat': betahat,
        'residuals': residuals,
        'residuals_studentized': residuals_studentized,
        's2': s2_ols,
        'R2': R2_ols,
        'R2_adj': R2_adj_ols,
        'cov_beta': cov_beta_ols,
        'se_beta': se_beta_ols,
        't_stats': t_stats_ols,
        'p_value': p_value_ols,
        'conf_int_95': ci_asymp_ols_95,
    }

    # --- 7️⃣ Compute Newey–West quantities ---
    cov_nw = cov_hac(model, nlags=bandwidth)
    se_nw = np.sqrt(np.diag(cov_nw))
    t_nw = betahat / se_nw
    p_nw = 2 * (1 - norm.cdf(np.abs(t_nw)))
    ci_nw_95 = np.column_stack([betahat - z_95*se_nw, betahat + z_95*se_nw])

    results[f'NW'] = {
        'lag': bandwidth,
        'var_sandwich': cov_nw,
        'cov_beta': np.diag(cov_nw),
        'conf_int_95': ci_nw_95,
        'p_value': p_nw
    }

    return results

In [307]:
results = one_replication(T, betas_true, bandwidth, omega_garch, alpha_garch, beta_garch)

In [308]:
np.set_printoptions(formatter={'float_kind': '{:0.3}'.format})
results["OLS"]["conf_int_95"]

array([[-0.0459, 0.0655],
       [1.0, 1.12],
       [0.496, 0.615],
       [-0.562, -0.447]])

In [309]:
print(results[f"NW"]["conf_int_95"])

[[-0.0414 0.061]
 [0.991 1.13]
 [0.498 0.613]
 [-0.546 -0.462]]


In [310]:
np.set_printoptions(formatter={'float_kind': '{:0.3e}'.format})
results["OLS"]["p_value"]

array([7.306e-01, 0.000e+00, 0.000e+00, 0.000e+00])

In [311]:
print(results[f"NW"]["p_value"])

[7.078e-01 0.000e+00 0.000e+00 0.000e+00]


In [312]:
residuals = results['OLS']['residuals']
residuals_studentized = results['OLS']['residuals_studentized']

# --- QQ plot (Plotly) ---
qq_theoretical = np.sort(stats.norm.rvs(size=T))
qq_empirical = np.sort(residuals_studentized)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=qq_theoretical,
    y=qq_empirical,
    mode='markers',
    name='Studentized residuals',
    marker=dict(color='rgba(50,100,200,0.6)')
))

# 45° reference line
min_val = min(qq_theoretical.min(), qq_empirical.min())
max_val = max(qq_theoretical.max(), qq_empirical.max())
fig.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    name='Normal line',
    line=dict(color='red', dash='dash')
))

fig.update_layout(
    title=dict(text="QQ Plot of Studentized Residuals", x=0.5, xanchor='center'),
    xaxis_title="Theoretical Quantiles (Normal)",
    yaxis_title="Empirical Quantiles (Studentized Residuals)",
    width=800,
    height=600,
    template='plotly_white',
    legend=dict(
        font=dict(size=14),
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='center',
        x=0.5
    )
)

fig.show()

In [313]:
# Test visually if the residuals can be clustered
residuals = results['OLS']['residuals']
T = len(residuals)
time_idx = np.arange(T)

# Create subplot: scatter (colored by time) + histogram
fig = make_subplots(rows=1, cols=2, subplot_titles=["Residuals over time (colored by time)", "Residuals histogram"], horizontal_spacing=0.12)

fig.add_trace(
    go.Scatter(
        x=time_idx,
        y=residuals,
        mode='markers',
        marker=dict(
            color=time_idx,
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title='Time'),
            size=0,
            opacity=0.85
        ),
        name='',
        hovertemplate='t=%{x}<br>res=%{y:.4f}<extra></extra>'
    ),
)

fig.add_trace(
    go.Histogram(
        x=residuals,
        nbinsx=30,
        marker=dict(color='rgba(50,100,200,0.7)'),
        name='Residuals histogram',
        showlegend=False
    ),
    row=1, col=2
)

# optional: add a rug / KDE overlay on the histogram for detail
fig.add_trace(
    go.Histogram(
        x=residuals,
        nbinsx=30,
        histnorm='probability density',
        marker=dict(color='rgba(0,0,0,0)'),
        showlegend=False,
        opacity=0.0,
    ),
    row=1, col=2
)

fig.update_layout(
    title_text='Residuals: time-colored scatter and histogram',
    width=1000,
    height=450,
    template='plotly_white'
)

fig.update_xaxes(title_text='Time index', row=1, col=1)
fig.update_yaxes(title_text='Residual', row=1, col=1)
fig.update_xaxes(title_text='Residual value', row=1, col=2)
fig.update_yaxes(title_text='Count', row=1, col=2)

fig.show()

In [314]:
# --- Normality tests for different values of sample sizes T ---
sample_sizes = [100, 250, 500, 1000, 2000]

for T in sample_sizes:
    results = one_replication(T, betas_true, bandwidth, omega_garch, alpha_garch, beta_garch)
    residuals = results['OLS']['residuals']
    residuals_studentized = results['OLS']['residuals_studentized']

    # --- Normality tests ---
    # Jarque–Bera test
    jb_stat, jb_p = stats.jarque_bera(residuals_studentized)

    # Shapiro–Wilk test
    sw_stat, sw_p = stats.shapiro(residuals_studentized)

    # Kolmogorov–Smirnov test
    ks_stat, ks_p = stats.kstest(residuals_studentized, 'norm')

    # --- Collect results ---
    tests_results = {
        'Jarque–Bera': {
            'statistic': jb_stat,
            'p_value': jb_p,
            'reject_null': jb_p < 0.05,
            'sample_size': T
        },
        'Shapiro–Wilk': {
            'statistic': sw_stat,
            'p_value': sw_p,
            'reject_null': sw_p < 0.05,
            'sample_size': T
        },
        'Kolmogorov–Smirnov': {
            'statistic': ks_stat,
            'p_value': ks_p,
            'reject_null': ks_p < 0.05,
            'sample_size': T
        }
    }
    display(pd.DataFrame(tests_results).T)


,statistic,p_value,reject_null,sample_size
Jarque–Bera,9.35363,0.009309,True,100
Shapiro–Wilk,0.975563,0.059726,False,100
Kolmogorov–Smirnov,0.082589,0.477517,False,100


,statistic,p_value,reject_null,sample_size
Jarque–Bera,0.986327,0.610691,False,250
Shapiro–Wilk,0.994457,0.495072,False,250
Kolmogorov–Smirnov,0.028572,0.983341,False,250


,statistic,p_value,reject_null,sample_size
Jarque–Bera,4.331229,0.114679,False,500
Shapiro–Wilk,0.995866,0.213932,False,500
Kolmogorov–Smirnov,0.026845,0.854173,False,500


,statistic,p_value,reject_null,sample_size
Jarque–Bera,13.83419,0.000991,True,1000
Shapiro–Wilk,0.996111,0.013148,True,1000
Kolmogorov–Smirnov,0.021286,0.74697,False,1000


,statistic,p_value,reject_null,sample_size
Jarque–Bera,339.507032,0.0,True,2000
Shapiro–Wilk,0.983336,0.0,True,2000
Kolmogorov–Smirnov,0.036221,0.010256,True,2000


In [315]:
def montecarlo_analysis(T,omega_garch, alpha_garch, beta_garch, betas, bandwidth, n_rep=10000):
    """Run Monte Carlo to check CI coverage and p-values"""
    p = len(betas)
    betahat_distr = np.zeros((n_rep,p))

    for i in trange(n_rep):
        res = one_replication(T, betas, bandwidth, omega_garch, alpha_garch, beta_garch)
        betahat_distr[i,:] = res['OLS']['betahat']
    return np.array(betahat_distr)

# Example usage
T = 200
bandwidth = int(4 * (T/100)**(2/9))

betahat_distr = montecarlo_analysis(T, omega_garch, alpha_garch, beta_garch, betas_true, bandwidth)
np.set_printoptions(formatter={'float_kind': '{:0.3}'.format})
print(betahat_distr)

  0%|          | 0/10000 [00:00<?, ?it/s]

[[0.0106 1.03 0.497 -0.511]
 [-0.0389 1.02 0.501 -0.51]
 [-0.0159 1.04 0.463 -0.409]
 ...
 [-0.0385 0.989 0.513 -0.489]
 [-0.0304 0.978 0.511 -0.553]
 [-0.021 1.02 0.543 -0.535]]


In [316]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

k = betahat_distr.shape[1]
n_rows, n_cols = 2, 2

# --- Compute global y-axis limit for consistent density scale ---
all_counts = []
for i in range(k):
    hist, _ = np.histogram(betahat_distr[:, i], bins=100, density=True)
    all_counts.append(np.max(hist))
y_max = max(all_counts) * 1.1

# --- Create subplots ---
fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f"β{i}" for i in range(k)],
    horizontal_spacing=0.12,
    vertical_spacing=0.15
)

# --- Add each β̂ histogram + vertical lines ---
for i in range(k):
    row = i // n_cols + 1
    col = i % n_cols + 1
    beta_true = betas_true[i]
    beta_mean = np.mean(betahat_distr[:, i])

    # Histogram
    fig.add_trace(
        go.Histogram(
            x=betahat_distr[:, i],
            nbinsx=100,
            histnorm='probability density',
            marker_color='rgba(0, 90, 180, 0.6)',
            showlegend=False
        ),
        row=row, col=col
    )

    # Vertical lines: true β (green) and sample mean (red dashed)
    fig.add_vline(
        x=beta_true,
        line=dict(color='green', width=3),
        row=row, col=col
    )
    fig.add_vline(
        x=beta_mean,
        line=dict(color='red', width=2, dash='dash'),
        row=row, col=col
    )

# --- Add dummy traces for legend ---
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
                         line=dict(color='green', width=3),
                         name='True β'))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
                         line=dict(color='red', width=2, dash='dash'),
                         name='Sample Mean β_OLS'))

# --- Style adjustments ---
fig.update_xaxes(title_text="β_OLS value", title_font=dict(size=14))
fig.update_yaxes(range=[0, y_max], title_text="Density", title_font=dict(size=14))

fig.update_layout(
    height=900,
    width=1150,
    title={
        'text': "Monte Carlo Distributions of β_OLS Estimates",
        'x': 0.5,  # center title
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=22, family="Arial Bold")
    },
    template='plotly_white',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.18,
        xanchor="center",
        x=0.5,
        font=dict(size=15, family="Arial", color="black"),
        bgcolor="rgba(255,255,255,0.6)",
        bordercolor="lightgray",
        borderwidth=1
    ),
    margin=dict(l=70, r=70, t=100, b=100)
)

fig.show()


In [317]:
# --- Monte Carlo test of empirical size at 1%, 5%, and 10% ---
print(results['NW']['p_value'])

n_rep = 1000
alpha_levels = [0.01, 0.05, 0.10]

# Initialize counters: one row per alpha, one column per beta
counter_ols = np.zeros((len(alpha_levels), len(betas_true)))
counter_nw  = np.zeros((len(alpha_levels), len(betas_true)))

for _ in trange(n_rep, desc="Monte Carlo replications"):
    results = one_replication(T, betas_true, bandwidth, omega_garch, alpha_garch, beta_garch)
    p_value_ols = np.array(results['OLS']['p_value'])
    p_value_nw  = np.array(results['NW']['p_value'])

    for j, alpha in enumerate(alpha_levels):
        counter_ols[j] += (p_value_ols < alpha).astype(int)
        counter_nw[j]  += (p_value_nw < alpha).astype(int)

# Compute empirical rejection frequencies
reject_rate_ols = counter_ols / n_rep
reject_rate_nw  = counter_nw / n_rep


[0.578 0.0 0.0 0.0]


Monte Carlo replications:   0%|          | 0/1000 [00:00<?, ?it/s]

In [318]:
# turn into DataFrames for better display
df_ols = pd.DataFrame(reject_rate_ols, index=[f'α={a}' for a in alpha_levels],
                      columns=[f'β{i}' for i in range(len(betas_true))])
df_nw  = pd.DataFrame(reject_rate_nw, index=[f'α={a}' for a in alpha_levels],
                      columns=[f'β{i}' for i in range(len(betas_true))])
print("Empirical rejection rates (OLS):")
display(df_ols)
print("Empirical rejection rates (Newey–West):")
display(df_nw)

Empirical rejection rates (OLS):


,β0,β1,β2,β3
α=0.01,0.006,1.0,1.0,1.0
α=0.05,0.037,1.0,1.0,1.0
α=0.1,0.085,1.0,1.0,1.0


Empirical rejection rates (Newey–West):


,β0,β1,β2,β3
α=0.01,0.007,1.0,1.0,1.0
α=0.05,0.045,1.0,1.0,1.0
α=0.1,0.084,1.0,1.0,1.0
